# Charleston, SC

This guide builds a multimodal network for the Charleston, South Carolina metropolitan area from real data, then uses it to measure how far a rider can get from downtown.

It downloads GTFS feeds through Transitland and streets from OpenStreetMap, so running it needs network access and a free [Transitland API key](https://www.transit.land), set as the `TRANSITLAND_API_KEY` environment variable.

In [ ]:
import geodatasets
import geopandas as gpd
import pandarm as pdna
import pandas as pd

import netweave as nw

## Study area

`geodatasets` provides census tracts for the metropolitan area, and their outline bounds everything that follows.

In [ ]:
charleston_gdf = gpd.read_file(geodatasets.get_path("geoda charleston1"))
ax = charleston_gdf.plot(figsize=(6, 6), edgecolor="white", linewidth=0.3)
ax.set_axis_off()

## Feeds

`fetch_feeds` finds the feeds with stops in the area, downloads them, and merges them. Every feed's IDs gain a prefix from its Onestop ID, so IDs from different agencies cannot collide. A feed that cannot be used is left out with a warning and recorded in `feed.missing`. To see what is available without downloading anything, use `search_feeds`.

In [ ]:
feed = nw.fetch_feeds(polygon=charleston_gdf)

print(f"{len(feed.stops):,} stops, {len(feed.routes):,} routes, {len(feed.trips):,} trips")
print(f"Left out: {feed.missing}")

## Transit network

A transit network represents one time window, here the morning peak on the busiest Monday in each feed. It has a node for each stop and for each route at a stop. Ride edges cost the mean scheduled time between stops, and boarding edges cost half the mean headway. `metadata` records the date each feed used.

In [ ]:
transit = nw.create_transit_network(
    feed,
    day="monday",
    timerange=("07:00:00", "10:00:00"),
    prune=charleston_gdf,
)

pd.DataFrame(transit.metadata["feeds"])[["prefix", "service_date"]]

In [ ]:
ride_edges = transit.edges[transit.edges["edge_type"] == "ride"]
ax = ride_edges.plot(figsize=(6, 6), linewidth=0.5)
ax.set_axis_off()

## Pedestrian network

`pandarm` downloads the streets. The impedance of this network is length in meters, which `integrate` converts to walking time.

In [ ]:
pedestrian = pdna.Network.from_gdf(charleston_gdf)

## Integration

`integrate` joins each stop to its nearest pedestrian node and returns a `pandarm.Network` whose single impedance, `minutes`, covers walking, waiting and riding. The node and edge tables gain labels such as `node_type` and `edge_type`, and `net.metadata` records what went into the network.

In [ ]:
net = nw.integrate(transit, pedestrian, max_connector_dist=500)

In [ ]:
waits = net.edges_df.loc[net.edges_df["edge_type"] == "board", "minutes"]

print(f"Stops beyond walking reach: {net.metadata['stranded_stops']}")
print(f"Median wait to board: {waits.median():.0f} minutes")
net.edges_df["edge_type"].value_counts()

## Analysis

The result is ready for accessibility analysis in `pandarm`. Pedestrian nodes keep the IDs they had in `pedestrian`, and points given to `get_node_ids` and `set_pois` snap only to pedestrian nodes. Query results cover every node, so filter them on `node_type`.

How much transit adds depends on the time budget, because a transit trip spends much of that budget walking to a stop and waiting to board. Here is the number of street nodes a rider can reach from downtown Charleston within several budgets, on foot and with transit:

In [ ]:
downtown = gpd.GeoSeries(gpd.points_from_xy([-79.9311], [32.7765]), crs=4326).to_crs(net.crs)
origin = net.get_node_ids(downtown.x, downtown.y).iloc[0]
meters_per_minute = 1 / nw.transform.walk_minutes(1)


def streets_within(minutes):
    """Return the street nodes reachable within a budget, with their travel times."""
    reach = net.nodes_in_range([origin], minutes, imp_name="minutes")
    nodes = net.nodes_df.loc[reach["destination"]].assign(minutes=reach["minutes"].to_numpy())
    return nodes[nodes["node_type"] == "pedestrian"]


budgets = pd.DataFrame({"minutes": [30, 45, 60, 90]})
budgets["on foot"] = [
    len(pedestrian.nodes_in_range([origin], m * meters_per_minute, imp_name="length"))
    for m in budgets["minutes"]
]
budgets["with transit"] = [len(streets_within(m)) for m in budgets["minutes"]]
budgets["gain"] = (budgets["with transit"] / budgets["on foot"]).round(2)
budgets

Every street node within 90 minutes of downtown, colored by travel time:

In [ ]:
ax = streets_within(90).plot(
    column="minutes",
    cmap="viridis_r",
    markersize=0.5,
    legend=True,
    legend_kwds={"label": "Minutes from downtown"},
    figsize=(6, 6),
)
ax.set_axis_off()